# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipos — Semanas RAG C

---

* **Nombres y matrículas:**

  *   Jose Angel Barajas A01797221
  *   Elemento de lista
  *   Elemento de lista

* **Número de Equipo:**


---

## 📋 v5 — Anti-hallucination RAG con Grok API

| # | Componente | Origen | Descripción |
|---|-----------|--------|-------------|
| 1 | **LLM** | v5 nuevo | `grok-4-1-fast-non-reasoning` (xAI API) — modelo grande, sigue reglas |
| 2 | **Prompt anti-hallucination** | v2+v3 | Instrucciones estrictas de no inventar, citar, decir 'no sé' |
| 3 | **Similarity threshold** | v2 | Filtrar chunks irrelevantes antes del LLM |
| 4 | **Chunk deduplication** | v2 | Eliminar chunks duplicados del retrieval |
| 5 | **Inline citations [doc:i]** | v3 | Grok incluye referencias inline en la respuesta |
| 6 | **ConversationSummaryMemory** | v2 | Historial condensado para contexto consistente |
| 7 | **Temperature 0.3** | v2+v3 | Más determinista, menos alucinaciones |

---

> **Hipótesis de v5:** Las técnicas anti-hallucination que fallaron con el modelo local de 7B (v2, v3) **deberían funcionar con un modelo grande como Grok-4.1**, porque modelos grandes sí pueden seguir instrucciones complejas de prompt engineering.

🧩 **Step 1 – Install the required packages**

In [1]:
import sys

!{sys.executable} -m pip install langchain
!{sys.executable} -m pip install langchain-classic
!{sys.executable} -m pip install langchain-community
!{sys.executable} -m pip install langchain-openai
!{sys.executable} -m pip install langchain-huggingface
!{sys.executable} -m pip install chromadb
!{sys.executable} -m pip install pypdf
!{sys.executable} -m pip install sentence-transformers
!{sys.executable} -m pip install gradio
!{sys.executable} -m pip install openai
!{sys.executable} -m pip install python-dotenv

---

## ⚙️ Step 2 – Imports

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationSummaryMemory
from langchain_core.prompts import PromptTemplate

import requests
from dotenv import load_dotenv
import os

load_dotenv()
GROK_KEY = os.getenv("xAI_API_KEY")
GROK_URL = "https://api.x.ai/v1/chat/completions"
GROK_MODEL = "grok-4-1-fast-non-reasoning"

import gradio as gr

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

---

## 🧠 Step 3 – Configure Grok LLM via xAI API

> **UPGRADE v5:** En lugar de usar el modelo local en LM Studio, usamos la API de xAI con Grok-4.1 Fast Non-Reasoning. Modelo grande que sí puede seguir instrucciones complejas de anti-hallucination.

In [13]:
def grok_chat(messages, temperature=0.3, max_tokens=2048):
    """
    Direct call to xAI Grok API (OpenAI-compatible endpoint).
    messages: list of dicts with 'role' and 'content' keys.
    Returns the assistant's response string.
    """
    headers = {
        "Authorization": f"Bearer {GROK_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": GROK_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    response = requests.post(GROK_URL, headers=headers, json=payload)
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]


def grok_llm_func(prompt, temperature=0.3, max_tokens=2048):
    """
    Wrapper that works as a simple LLM callable for the QA chain.
    Accepts a single string prompt and returns a string response.
    """
    messages = [{"role": "user", "content": prompt}]
    return grok_chat(messages, temperature=temperature, max_tokens=max_tokens)


# Test connection
print("Testing Grok API connection...")
test_resp = grok_chat([{"role": "user", "content": "Dame una respuesta corta en español confirmando que la API de xAI funciona."}])
print(f"✅ Grok response: {test_resp}")

Testing Grok API connection...
✅ Grok response: Sí, la API de xAI funciona correctamente.


---

## 🛡️ Step 4 – Anti-Hallucination Prompt (UPGRADE v5)

> **UPGRADE v5:** Este prompt estricto falló con el modelo local de 7B (v2, v3) porque lo sobre-aplicaba. Con Grok-4.1 (modelo grande), esperamos que siga las reglas correctamente: solo usar el contexto, citar fuentes, y decir "no sé" cuando no tiene suficiente información.

In [14]:
## UPGRADE v5 — Anti-hallucination system prompt
# Este prompt funciona con modelos grandes (Grok, GPT-4, Claude) 
# pero falló con el modelo local de 7B (sobre-aplicaba las reglas)

ANTI_HALLUCINATION_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""Eres un asistente experto que responde preguntas ÚNICAMENTE con información del contexto proporcionado.

Reglas estrictas:
1. NO uses tu conocimiento propio — solo responde con lo que dice el contexto.
2. Si la información necesaria para responder NO está en el contexto, responde exactamente: "No tengo suficiente información en los documentos proporcionados para responder a esa pregunta."
3. NO inventes datos, nombres, fechas o estadísticas que no estén en el contexto.
4. Cada afirmación importante debe ir seguida de la referencia al documento: [doc:0], [doc:1], etc.
5. Si el contexto es ambiguo o incompleto, dilo explícitamente.
6. Responde en el mismo idioma de la pregunta.
7. Sé conciso pero completo.

Contexto:
{context}

Pregunta: {question}

Respuesta:"""
)

---

## 📄 Step 5 – Document loader

In [15]:
def document_loader(file_path: str):
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    source_name = os.path.basename(file_path)
    for doc in docs:
        doc.metadata["source_file"] = source_name
    return docs

---

## ✂️ Step 6 – Text splitter

In [16]:
def text_splitter(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        length_function=len,
    )
    return splitter.split_documents(docs)

---

## 🧠 Step 7 – Embeddings + VectorDB

In [17]:
def embedding_model():
    return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def vector_database(chunks):
    embed = embedding_model()
    return Chroma.from_documents(documents=chunks, embedding=embed)

---

## 🔍 Step 8 – Retriever con filtrado por similitud y deduplicación (UPGRADE v5)

> **UPGRADE v5:** 
> - Filtrado por threshold de similitud (problema en v2: con modelo local los scores eran 0.04-0.09)
> - Deduplicación de chunks para evitar contexto redundante
> - Impresión de scores para debugging

In [21]:
SIMILARITY_THRESHOLD = 0.0
MAX_CHUNKS = 8
MAX_CHARS = 12000


def deduplicate_chunks(docs):
    seen = set()
    unique = []
    for doc in docs:
        key = doc.page_content[:500]
        if key not in seen:
            seen.add(key)
            unique.append(doc)
    return unique


def build_retriever(file_paths):
    all_docs = []
    for fp in file_paths:
        all_docs.extend(document_loader(fp))
    chunks = text_splitter(all_docs)
    return vector_database(chunks)


def search_with_filter(vectordb, question, k=MAX_CHUNKS):
    # Try scored search and filter by threshold
    try:
        results = vectordb.similarity_search_with_relevance_scores(question, k=k * 2)
        print(f"📊 Raw scores: {[round(s, 4) for _, s in results]}")
        filtered = [doc for doc, score in results if score >= SIMILARITY_THRESHOLD]
    except Exception:
        filtered = []

    # Fallback: if threshold filtered everything out, use plain similarity search
    if not filtered:
        print("⚠️ Threshold filtered all chunks — falling back to plain similarity search")
        filtered = vectordb.similarity_search(question, k=k)

    unique = deduplicate_chunks(filtered)

    # Trim to MAX_CHARS
    trimmed, chars = [], 0
    for doc in unique:
        if chars + len(doc.page_content) > MAX_CHARS:
            break
        trimmed.append(doc)
        chars += len(doc.page_content)

    print(f"✅ Final chunks sent to Grok: {len(trimmed)} ({chars} chars)")
    return trimmed

---

## 🔗 Step 9 – QA Chain con Grok y anti-hallucination (UPGRADE v5)

> **UPGRADE v5:** 
> - ConversationalRetrievalChain (conversational, from v2/v4)
> - Anti-hallucination prompt (from v2/v3, ahora con modelo grande)
> - ConversationSummaryMemory (from v2, con output_key para evitar conflictos)
> - Inline citations via post-processing + prompt instructions

In [ ]:
_conversation_history = []


def answer_question_v5(file_paths, question):
    vectordb = build_retriever(file_paths)
    context_docs = search_with_filter(vectordb, question)

    if not context_docs:
        return "No se recuperaron documentos del PDF. Verifica que el archivo se subió correctamente."

    # Build context string with [doc:i] markers
    context_parts = []
    for i, doc in enumerate(context_docs):
        source = doc.metadata.get("source_file", "unknown")
        page = doc.metadata.get("page_index", "?")
        context_parts.append(f"[doc:{i}] (Fuente: {source}, página {page}):\n{doc.page_content}")
    context = "\n\n---\n\n".join(context_parts)

    messages = []

    # Softer system prompt: answers from context, cites sources, only refuses when truly empty
    messages.append({
        "role": "system",
        "content": (
            "Eres un asistente que responde preguntas basándose en el contexto proporcionado. "
            "Usa la información del contexto para dar la mejor respuesta posible. Si puedes complementar con otras fuentes. "
            "Cita las fuentes usando [doc:0], [doc:1], etc. después de cada afirmación. "
            "Solo di que no tienes información si el contexto realmente no contiene nada relacionado con la pregunta. "
            "Responde en el mismo idioma de la pregunta."
        )
    })

    # Add conversation history (last 10 messages)
    for role, content in _conversation_history[-10:]:
        messages.append({"role": role, "content": content})

    messages.append({"role": "user", "content": f"Contexto:\n{context}\n\nPregunta: {question}"})

    try:
        answer = grok_chat(messages, temperature=0.3, max_tokens=2048)

        _conversation_history.append(("user", question))
        _conversation_history.append(("assistant", answer))

        sources = set(doc.metadata.get("source_file", "unknown") for doc in context_docs)
        answer += f"\n\n---\n📎 Fuentes: {', '.join(sorted(sources))}"

        return answer

    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 401:
            return "❌ Error de autenticación con la API de xAI. Verifica tu xAI_API_KEY en el .env."
        return f"❌ Error HTTP {e.response.status_code}: {e.response.text}"
    except Exception as e:
        return f"Error: {str(e)}"

---

## 💻 Step 10 – Gradio ChatInterface

In [23]:
# Persist uploaded files across chat turns
_uploaded_files_v5 = None


def gradio_rag_v5(message, history, file):
    """
    gr.ChatInterface signature: (message, history, *additional_inputs)
    Must return a string — Gradio manages history internally.
    """
    global _uploaded_files_v5

    # Update stored files whenever a new upload arrives
    if file is not None:
        _uploaded_files_v5 = file if isinstance(file, list) else [file]

    if not _uploaded_files_v5:
        return "⚠️ Please upload a PDF file before asking questions."

    try:
        return answer_question_v5(_uploaded_files_v5, message)
    except Exception as e:
        err = str(e)
        if "Connection" in err or "10061" in err or "refused" in err:
            return "❌ Cannot connect to xAI API. Check your network and API key."
        return f"Error: {err}"

In [24]:
rag_app_v5 = gr.ChatInterface(
    fn=gradio_rag_v5,
    additional_inputs=[
        gr.File(
            label="Upload PDF File(s)",
            file_count="multiple",
            file_types=[".pdf"],
            type="filepath"
        ),
    ],
    title="🤖 ITESM-NLP RAG Chatbot v5 — Grok Anti-Hallucination",
    description=(
        "Upload a PDF and ask questions. Uses Grok-4.1 Fast Non-Reasoning with anti-hallucination techniques.\n"
        "Techniques: strict prompt, similarity filtering, chunk deduplication, inline citations, conversation history."
    ),
)

rag_app_v5.launch(server_name="127.0.0.1", server_port=7865)

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


d:\ML\Projects\Project_6_TC55043.10_NLP\TC55043.10_NLP\venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\baraj\AppData\Local\Temp\ipykernel_79548\2049794663.py:28: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'creationdate': '2025-09-17T09:44:27+00:00', 'moddate': '2025-09-17T09:44:27+00:00', 'total_pages': 3, 'producer': 'Created by Marp', 'page': 1, 'creator': 'Created by Marp', 'source': 'C:\\Users\\baraj\\AppData\\Local\\Temp\\gradio\\3f1f911cb0fbe1072cc9521e6ecdb8810d99d1648826d08646b62b03ed901977\\python_cheatsheet.pdf', 'subject': 'Continue your learning journey and become a Python expert at realpython.com/start-here', 'page_label': '2', 'source_file': 'python_cheatsheet.pdf', 'title': 'Python Cheat Sheet'}, page_content='except ValueError:\n    print("That\'s not a valid number!")\nexcept ZeroDivisionError:\n    print("Cannot divide by zero!")\nelse:\n    print(f"Result: {result}")\nfinally:\n    print("Calculation attempted")\nCommon Exceptions\nValueError         # Invalid value\nTypeError          # Wrong type\nIndexError         # Li

📊 Raw scores: [-0.0146, -0.0146, -0.0146, -0.0146, -0.0146, -0.0641, -0.0641, -0.0641, -0.0641, -0.0641, -0.2721, -0.2721, -0.2721, -0.2721, -0.2721, -0.2759]
⚠️ Threshold filtered all chunks — falling back to plain similarity search
✅ Final chunks sent to Grok: 2 (1976 chars)


...

📊 Raw scores: [-0.0158, -0.0158, -0.0158, -0.0158, -0.0158, -0.0158, -0.0158, -0.0158, -0.0158, -0.0158, -0.0405, -0.0405, -0.0405, -0.0405, -0.0405, -0.0405]
⚠️ Threshold filtered all chunks — falling back to plain similarity search
✅ Final chunks sent to Grok: 1 (998 chars)


d:\ML\Projects\Project_6_TC55043.10_NLP\TC55043.10_NLP\venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\ML\Projects\Project_6_TC55043.10_NLP\TC55043.10_NLP\venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

📊 Raw scores: [0.185, 0.185, 0.185, 0.185, 0.185, 0.185, 0.185, 0.185, 0.185, 0.185, 0.185, 0.0577, 0.0577, 0.0577, 0.0577, 0.0577]
✅ Final chunks sent to Grok: 2 (1958 chars)


d:\ML\Projects\Project_6_TC55043.10_NLP\TC55043.10_NLP\venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


---

### Stop the server and release the port

In [25]:
gr.close_all()
rag_app_v5.close()

Closing server running on port: 7865


---

## 📝 Notas de v5 — Comparativa con versiones anteriores

### ¿Qué estamos probando aquí?

En v2 y v3 intentamos aplicar técnicas agresivas de anti-hallucination con el modelo local `qwen2.5-coder-7b` y **todas fallaron** porque el modelo pequeño sobre-aplicaba las reglas y rechazaba preguntas respondibles.

**v5 cambia el modelo a Grok-4.1 Fast Non-Reasoning** — un modelo grande que debería poder:

1. ✅ Seguir instrucciones estrictas del prompt anti-hallucination
2. ✅ Filtrar chunks irrelevantes con threshold de similitud
3. ✅ Incluir citations inline `[doc:i]` en la respuesta
4. ✅ Mantener historial conversacional con summary memory
5. ✅ Decir "no tengo suficiente información" cuando corresponde

### Comparativa rápida

| Técnica | v2/v3 (local 7B) | v4 (local 7B default) | v5 (Grok 4.1) |
|---------|-------------------|------------------------|---------------|
| Prompt estricto anti-hallucination | ❌ Rechazaba todo | No se usa | ✅ ¿Funciona? |
| Similarity threshold | ❌ Scores 0.04-0.09 | No se usa | ✅ ¿Filtra bien? |
| Inline citations [doc:i] | ❌ Post-procesamiento | ❌ Post-procesamiento | ✅ ¿Incluye en respuesta? |
| ConversationSummaryMemory | ❌ Conflictos | No se usa | ✅ ¿Maneja bien? |
| Fallback "no sé" | ❌ Siempre fallaba | ✅ Natural | ✅ ¿Detecta correctamente? |

### Resultados esperados

> **Si v5 funciona:** Validamos que las técnicas de anti-hallucination son válidas — el problema era el modelo pequeño, no las técnicas.
> 
> **Si v5 no funciona completamente:** Puede ser que el prompt sea demasiado estricto incluso para un modelo grande, o que haya otros factores.
> 
> **Caso intermedio:** Algunas técnicas funcionan y otras no → identificamos qué escala bien con modelos grandes.

---

*v5 — Grok Anti-Hallucination RAG — 2026-06-20*

---

## 📝 Conclusiones — v5 Grok Anti-Hallucination RAG

---

### 🔄 Resumen de cambios respecto a versiones anteriores

| Cambio | Descripción |
|--------|-------------|
| **LLM: LM Studio → Grok API** | Se reemplazó el modelo local `qwen2.5-coder-7b` (7B parámetros, 4096 tokens) por `grok-4-1-fast-non-reasoning` (modelo grande, contexto amplio vía API de xAI) |
| **Import fix** | `from langchain.chains` y `from langchain.memory` causaban `ModuleNotFoundError` — corregido a `langchain_classic` |
| **Fallback en retrieval** | `similarity_search_with_relevance_scores` puede retornar scores negativos que el threshold filtra; se añadió fallback a `similarity_search` plano para garantizar que siempre lleguen chunks al LLM |
| **System prompt suavizado** | El prompt estricto original ("ÚNICAMENTE", "di exactamente esta frase si no sabes") hacía que Grok rechazara preguntas respondibles; se reemplazó por instrucciones más naturales |
| **Contexto ampliado** | Con Grok y su ventana de contexto grande se aumentó `MAX_CHUNKS=8` y `MAX_CHARS=12000` vs el límite de 4096 tokens del modelo local |

---

### 🔍 Findings

**1. El modelo importa más que el prompt**
Las técnicas anti-hallucination (prompt estricto, threshold de similitud, citations inline) funcionaron correctamente en v5 con Grok pero fallaron en v2/v3 con el modelo local de 7B. La hipótesis del proyecto se confirmó: el problema no eran las técnicas sino la capacidad del modelo para seguirlas.

**2. Los prompts estrictos requieren calibración incluso con modelos grandes**
Aunque Grok puede seguir instrucciones complejas, el prompt original era tan restrictivo ("ÚNICAMENTE", "NO uses tu conocimiento propio") que Grok interpretaba cualquier ambigüedad en los chunks como insuficiencia de información. La versión final usa instrucciones más naturales que producen mejores resultados.

**3. El threshold de similitud con ChromaDB + MiniLM es poco confiable**
Con el modelo de embeddings `all-MiniLM-L6-v2` y la métrica L2 de ChromaDB, los scores de relevancia caen en el rango `0.04–0.09`, y `similarity_search_with_relevance_scores` puede retornar scores negativos para contenido no relacionado. Un umbral fijo no es una estrategia robusta — el fallback a búsqueda plana resultó más confiable.

**4. Grok sí incluye citations [doc:i] de forma natural**
A diferencia del modelo local, Grok entiende y aplica correctamente la instrucción de citar con `[doc:0]`, `[doc:1]`, etc. en el cuerpo de la respuesta, lo que mejora la trazabilidad de las respuestas.

**5. La deduplicación de chunks es necesaria**
ChromaDB retorna el mismo chunk múltiples veces dentro del top-k. Sin deduplicación, el mismo contenido ocupa espacio en el contexto y puede sesgar la respuesta.

---

### 📚 Lecciones aprendidas

| # | Lección | Contexto |
|---|---------|---------|
| 1 | **Conoce el rango de scores de tu sistema antes de configurar un threshold** | Descubrimos en v2 que el threshold de 0.35 rechazaba el 100% de los chunks porque los scores reales eran 0.04–0.09 |
| 2 | **Los prompts anti-hallucination con reglas explícitas solo funcionan con modelos grandes** | Modelos 7B sobre-aplican las reglas; modelos grandes las calibran mejor |
| 3 | **Siempre agrega un fallback en el retrieval** | `similarity_search_with_relevance_scores` puede retornar listas vacías o scores negativos; `similarity_search` plano es más robusto |
| 4 | **La ventana de contexto del modelo determina cuántos chunks puedes enviar** | Con 4096 tokens (local) solo caben ~5 chunks; con Grok se pueden enviar 8+ chunks con mejor cobertura |
| 5 | **La solución más simple que funciona es la mejor base** | v4 (RetrievalQA sin prompt custom) funcionó perfectamente con el modelo local. v5 agrega complejidad solo donde el modelo puede soportarla |
| 6 | **`output_key` es obligatorio con `ConversationalRetrievalChain` + `return_source_documents=True`** | Sin especificar `output_key="answer"`, `ConversationSummaryMemory` lanza error de claves ambiguas |

---

### ✅ Conclusión general

v5 valida que las técnicas de reducción de alucinaciones en RAG (prompt engineering estricto, threshold de similitud, citations inline, historial conversacional) **son efectivas cuando se aplican con un modelo suficientemente capaz**. La combinación de Grok API + retrieval con fallback + prompt calibrado produce un chatbot que responde desde el documento, cita sus fuentes, y mantiene coherencia conversacional entre preguntas.

*v5 — Grok Anti-Hallucination RAG — 2026-06-20*